In [42]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [43]:
data=pd.read_csv('netflix_titles_nov_2019.csv')

In [44]:
data.head(3)

,show_id,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,type
0,81193313,Chocolate,NaN,"Ha Ji-won, Yoon Kye-sang, Jang Seung-jo, Kang ...",South Korea,"November 30, 2019",2019,TV-14,1 Season,"International TV Shows, Korean TV Shows, Roman...",Brought together by meaningful meals in the pa...,TV Show
1,81197050,Guatemala: Heart of the Mayan World,"Luis Ara, Ignacio Jaunsolo",Christian Morales,NaN,"November 30, 2019",2019,TV-G,67 min,"Documentaries, International Movies","From Sierra de las Minas to Esquipulas, explor...",Movie
2,81213894,The Zoya Factor,Abhishek Sharma,"Sonam Kapoor, Dulquer Salmaan, Sanjay Kapoor, ...",India,"November 30, 2019",2019,TV-14,135 min,"Comedies, Dramas, International Movies",A goofy copywriter unwittingly convinces the I...,Movie


In [45]:
data.columns

Index(['show_id', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'type'],
      dtype='object')

In [46]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5837 entries, 0 to 5836
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       5837 non-null   int64 
 1   title         5837 non-null   object
 2   director      3936 non-null   object
 3   cast          5281 non-null   object
 4   country       5410 non-null   object
 5   date_added    5195 non-null   object
 6   release_year  5837 non-null   int64 
 7   rating        5827 non-null   object
 8   duration      5837 non-null   object
 9   listed_in     5837 non-null   object
 10  description   5837 non-null   object
 11  type          5837 non-null   object
dtypes: int64(2), object(10)
memory usage: 547.3+ KB


In [47]:
data.describe()

,show_id,release_year
count,5.837000e+03,5837.000000
mean,7.730079e+07,2013.688539
std,9.479777e+06,8.419088
min,2.698800e+05,1925.000000
25%,8.004520e+07,2013.000000
50%,8.016353e+07,2016.000000
75%,8.024188e+07,2018.000000
max,8.122720e+07,2020.000000


In [48]:
data.shape

(5837, 12)

# Data Preprocessing

In [49]:
data.isna().sum()

show_id            0
title              0
director        1901
cast             556
country          427
date_added       642
release_year       0
rating            10
duration           0
listed_in          0
description        0
type               0
dtype: int64

In [50]:
#data['director']=data['director'].fillna('Unknown Director')

#OR

data.fillna({
    'director': 'Unknown Director',
    'cast': 'Unknown Cast',
    'country': 'Unknown Country'
}, inplace=True)

In [51]:
data['date_added'] = pd.to_datetime(data['date_added'], errors='coerce')
data['date_missing'] = data['date_added'].isna().astype(int)

In [52]:
data.isna().sum()

show_id           0
title             0
director          0
cast              0
country           0
date_added      642
release_year      0
rating           10
duration          0
listed_in         0
description       0
type              0
date_missing      0
dtype: int64

In [53]:
data.drop('date_added', axis=1, inplace=True)

In [54]:
data['rating']=data['rating'].fillna(0)

In [55]:
data.isna().sum()

show_id         0
title           0
director        0
cast            0
country         0
release_year    0
rating          0
duration        0
listed_in       0
description     0
type            0
date_missing    0
dtype: int64

In [56]:
df_clean=data.copy()

## Usecase1: Popularity based recommendation system

### Recommend movies for a new user.

In [57]:
rating_map = {
    'TV-Y': 1,
    'TV-Y7': 2,
    'TV-G': 3,
    'G': 3,
    'PG': 4,
    'TV-PG': 5,
    'PG-13': 6,
    'TV-14': 7,
    'R': 8,
    'NC-17': 9,
    'TV-MA': 10
}


df_clean['rating_numeric'] = df_clean['rating'].map(rating_map)
df_clean['rating_numeric'] = df_clean['rating_numeric'].fillna(0)

In [58]:
# Generic recommendation when the new user log-in

top_movie_list = df_clean.groupby('title')['rating_numeric'].agg('max').sort_values(ascending=False)


In [59]:
Top_10_movies=top_movie_list[0:10]
Top_10_movies

title
St. Agatha                       10.0
OtherLife                        10.0
Functional Fitness               10.0
Furie                            10.0
Furthest Witness                 10.0
Operações Especiais              10.0
G.O.R.A                          10.0
GANTZ:O                          10.0
They’ll Love Me When I’m Dead    10.0
GHOUL                            10.0
Name: rating_numeric, dtype: float64

### Recommend movies for a new user based on country

In [60]:
# Recommend movie based on country login-in Example: Indian user with Indian Movies

india_df = df_clean[df_clean['country'].str.contains('India', na=False)]
Top_10_Indian_movies = (
    india_df.groupby('title')['rating_numeric']
    .max()
    .sort_values(ascending=False)
    .head(10)
)

Top_10_Indian_movies

title
Kabir Singh                      10.0
For Here or to Go?               10.0
Ek Hasina Thi                    10.0
Ek Khiladi Ek Haseena            10.0
Needhi Singh                     10.0
Tamanchey                        10.0
Nasha                            10.0
Naan Sigappu Manithan            10.0
Fashion                          10.0
Fear Files... Har Mod Pe Darr    10.0
Name: rating_numeric, dtype: float64

## Usecase2:Context Besed recommendation: 

### Movie Recommendations Based on User’s Previous Viewing Preferences


In [61]:
Horror_movie_list = df_clean[df_clean['listed_in'].str.contains('Horror', na=False)]

In [62]:
Top10_Horror_movie_list=Horror_movie_list.sort_values(by='rating_numeric',ascending=False).head(10)

In [63]:
Top10_Horror_movie_list

,show_id,title,director,cast,country,release_year,rating,duration,listed_in,description,type,date_missing,rating_numeric
2580,80189221,The Haunting of Hill House,Unknown Director,"Michiel Huisman, Carla Gugino, Timothy Hutton,...",United States,2018,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries","Flashing between past and present, a fractured...",TV Show,0,10.0
2772,80999985,Still,"Chatchai Katenut, Manussa Vorasingha, Tanwarin...","Mai Charoenpura, Akara Amarttayakul, Supakson ...",Thailand,2010,TV-MA,100 min,"Horror Movies, International Movies","A haunted hotel, a body in a water tank, a nig...",Movie,0,10.0
4323,80162143,The Forgotten,Oliver Frampton,"Clem Tibber, Elarica Johnson, James Doherty, S...",United Kingdom,2014,TV-MA,89 min,Horror Movies,After a teenager goes to live with his father ...,Movie,0,10.0
4299,80103311,The Bar,Álex de la Iglesia,"Blanca Suárez, Mario Casas, Carmen Machi, Secu...","Spain, Argentina",2017,TV-MA,102 min,"Comedies, Horror Movies, International Movies",Strangers holed up in a Madrid bar after witne...,Movie,0,10.0
2267,81001121,Gehenna: Where Death Lives,Hiroshi Katagiri,"Eva Swan, Simon Phillips, Justin Gordon, Doug ...","Japan, United States",2016,TV-MA,107 min,"Horror Movies, Thrillers",Developers looking for a location to build a r...,Movie,0,10.0
2367,81021831,Sabrina,Rocky Soraya,"Luna Maya, Christian Sugiono, Sara Wijayanto, ...",Indonesia,2018,TV-MA,114 min,"Horror Movies, International Movies",A toy manufacturer and his wife are terrorized...,Movie,0,10.0
4282,80128722,Gerald's Game,Mike Flanagan,"Carla Gugino, Bruce Greenwood, Henry Thomas, C...",United States,2017,TV-MA,103 min,"Horror Movies, Thrillers","When her husband's sex game goes wrong, Jessie...",Movie,0,10.0
2390,81030893,May the Devil Take You,Timo Tjahjanto,"Chelsea Islan, Pevita Pearce, Samo Rafael, Kar...",Indonesia,2018,TV-MA,111 min,"Horror Movies, International Movies",Hoping to find answers to her estranged father...,Movie,0,10.0
2407,70286533,Helix,Unknown Director,"Billy Campbell, Hiroyuki Sanada, Kyra Zagorsky...","United States, Canada",2015,TV-MA,1 Season,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",While investigating a possible outbreak at an ...,TV Show,0,10.0
2513,81000863,14 Cameras,"Scott Hussion, Seth Fuller","Neville Archambault, Amber Midthunder, Brytnee...",United States,2018,TV-MA,89 min,"Horror Movies, Thrillers","Upping the “13 Cameras” ante, this sequel find...",Movie,0,10.0


###  Using Weighted Average: 

In [64]:
df_clean.columns


Index(['show_id', 'title', 'director', 'cast', 'country', 'release_year',
       'rating', 'duration', 'listed_in', 'description', 'type',
       'date_missing', 'rating_numeric'],
      dtype='object')

In [65]:
# Create Score components if TV show,then 1 else 0 etc..
df_clean['type_score'] = (df_clean['type'] == 'TV Show').astype(int)
df_clean['romance_score'] = df_clean['listed_in'].str.contains('Romance', na=False).astype(int)

#Apply weights # give the weightage to column

df_clean['weighted_score'] = (
    df_clean['type_score'] * 0.4 +
    df_clean['romance_score'] * 0.6
)

# Filter the TV shows which have rating more than 8
filtered = df_clean[df_clean['rating_numeric'] > 8]


#Get recommendations
Top_recommendations = (
    filtered
    .sort_values(by='weighted_score', ascending=False)
    .drop_duplicates('title')
    .head(10)
)

In [66]:
Top_recommendations[:2]

,show_id,title,director,cast,country,release_year,rating,duration,listed_in,description,type,date_missing,rating_numeric,type_score,romance_score,weighted_score
2848,80220145,Undercover Law,Unknown Director,"Viña Machado, Juana del Río, Luna Baxter, Vale...",Colombia,2017,TV-MA,1 Season,"Crime TV Shows, International TV Shows, Spanis...",Female intelligence agents infiltrate the disp...,TV Show,0,10.0,1,0,0.4
2464,80217889,Follow This,Unknown Director,"John Stanton, Scaachi Koul, Azeen Ghorayshi, B...",United States,2018,TV-MA,3 Seasons,Docuseries,Follow the reporters at BuzzFeed as they probe...,TV Show,1,10.0,1,0,0.4


### Using Cosine Similarity: # Hybrid also using Rating

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Filter dataset

filtered_df = df_clean[
    (df_clean['type'] == 'TV Show') &
    (df_clean['listed_in'].str.contains('Comedy', na=False)) &
    (df_clean['country'].str.contains('United States', na=False))
].reset_index(drop=True)


#Create content
filtered_df['content'] = (
    filtered_df['listed_in'] + " " + filtered_df['description']
)

#TF-IDF

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(filtered_df['content'])

#Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix)

In [68]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
filtered_df['rating_scaled'] = scaler.fit_transform(filtered_df[['rating_numeric']])

In [69]:
def recommend_show(title, n=10):
    indices = pd.Series(filtered_df.index, index=filtered_df['title']).drop_duplicates()
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Combine similarity + rating
    hybrid_scores = []
    for i, score in sim_scores:
        final_score = (score * 0.7) + (filtered_df.loc[i, 'rating_scaled'] * 0.3)
        hybrid_scores.append((i, final_score))
    
    # Sort
    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    
    # Skip itself
    hybrid_scores = hybrid_scores[1:n+1]
    
    show_indices = [i[0] for i in hybrid_scores]
    
    return filtered_df.iloc[show_indices][['title', 'listed_in', 'rating_numeric']]

In [70]:
recommend_show("Tiffany Haddish Presents: They Ready")

,title,listed_in,rating_numeric
11,The Comedy Lineup,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
4,COMEDIANS of the world,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
20,Ari Shaffir: Double Negative,Stand-Up Comedy & Talk Shows,10.0
23,Dave Chappelle,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
18,Dave Chappelle: Equanimity & The Bird Revelation,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
17,The Standups,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
16,The Honeymoon Stand Up Special,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
22,Chelsea,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
13,The Break with Michelle Wolf,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0
10,Daniel Sloss: Live Shows,"Stand-Up Comedy & Talk Shows, TV Comedies",10.0


In [71]:
df_clean['rating_numeric'].value_counts()

rating_numeric
10.0    1937
7.0     1593
5.0      678
8.0      439
0.0      327
6.0      227
3.0      179
4.0      160
2.0      156
1.0      139
9.0        2
Name: count, dtype: int64